# Project 21: Flight Delay Prediction

## Task 4 — Chronological Split and Preprocessing

### Objective

The objective of this stage is to prepare the feature-engineered flight dataset for machine learning by validating the chronological splits and applying appropriate preprocessing techniques.

Since flight delay patterns can change over time, the data must be divided chronologically rather than randomly. This stage also ensures that preprocessing transformations are learned only from the training data, preventing information from the validation and test periods from influencing the training process.

The prepared datasets will be used in the subsequent modelling stages for Logistic Regression, XGBoost, and deep learning.

### 1. Loading the Feature-Engineered Dataset

We begin by importing the libraries required for data manipulation and preprocessing. The feature-engineered dataset generated in Task 3 is loaded, and the date column is converted to datetime format.

We inspect the dimensions, columns, missing values, target distribution, and existing split labels to ensure that the dataset is suitable for chronological splitting and preprocessing.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
import joblib

# Upload the latest complete feature-engineered CSV to Colab first.
DATA_PATH = Path('flights_feature_engineered_CORRECTED.csv')
assert DATA_PATH.is_file(), f'Upload the input CSV to: {DATA_PATH}'
df = pd.read_csv(DATA_PATH)
df['date'] = pd.to_datetime(df['date'], errors='raise')

# This is an export-row identifier, NOT a true original flight ID.
df['row_id'] = np.arange(len(df), dtype=np.int64)
assert len(df) == 327346, f'Unexpected input rows: {len(df)}'
assert df['date'].nunique() == 365
assert df['delayed'].notna().all()
assert set(df['split'].unique()) == {'train','val','test'}
print('Loaded:', DATA_PATH, df.shape)
display(df.head())

Loaded: flights_feature_engineered_CORRECTED.csv (327346, 25)


,date,split,delayed,carrier,origin,dest,route,distance,dep_time_sin,dep_time_cos,...,arr_time_cos,origin_flights_prev7d,origin_delay_rate_prev7d,carrier_flights_prev7d,carrier_delay_rate_prev7d,dest_flights_prev7d,dest_delay_rate_prev7d,route_flights_prev7d,route_delay_rate_prev7d,row_id
0,2013-01-01,train,0.0,UA,EWR,IAH,EWR_IAH,1400,0.980785,1.950903e-01,...,-0.569997,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0
1,2013-01-01,train,1.0,UA,LGA,IAH,LGA_IAH,1416,0.990866,1.348509e-01,...,-0.608761,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,1
2,2013-01-01,train,1.0,AA,JFK,MIA,JFK_MIA,1089,0.996195,8.715574e-02,...,-0.675590,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,2
3,2013-01-01,train,0.0,B6,JFK,BQN,JFK_BQN,1576,0.997859,6.540313e-02,...,-0.909961,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,3
4,2013-01-01,train,0.0,DL,LGA,ATL,LGA_ATL,762,1.000000,6.123234e-17,...,-0.632705,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,4


### 2. Chronological Train, Validation, and Test Splits

The project uses a chronological split to simulate predicting flight delays using historical data. The proposed training period is January to August 2013, the validation period is September to October 2013, and the test period is November to December 2013.

The training set is used to learn model parameters and preprocessing transformations. The validation set supports model selection and threshold tuning, while the test set is kept separate for final evaluation.

We first verify the date boundaries and flight counts for the existing split labels rather than performing a new random split.

In [ ]:
print("Dataset shape:", df.shape)

print("\nSplit labels and counts:")
print(df["split"].value_counts(dropna=False))

print("\nColumns:")
print(df.columns.tolist())

print("\nDate range:")
print(df["date"].min(), "to", df["date"].max())

print("\nMissing values by column:")
print(df.isna().sum()[df.isna().sum() > 0])

Dataset shape: (327346, 25)

Split labels and counts:
split
train    217727
val       55628
test      53991
Name: count, dtype: int64

Columns:
['date', 'split', 'delayed', 'carrier', 'origin', 'dest', 'route', 'distance', 'dep_time_sin', 'dep_time_cos', 'weekday_sin', 'weekday_cos', 'month_sin', 'month_cos', 'arr_time_sin', 'arr_time_cos', 'origin_flights_prev7d', 'origin_delay_rate_prev7d', 'carrier_flights_prev7d', 'carrier_delay_rate_prev7d', 'dest_flights_prev7d', 'dest_delay_rate_prev7d', 'route_flights_prev7d', 'route_delay_rate_prev7d', 'row_id']

Date range:
2013-01-01 00:00:00 to 2013-12-31 00:00:00

Missing values by column:
origin_delay_rate_prev7d     1759
carrier_delay_rate_prev7d    1768
dest_delay_rate_prev7d       1840
route_delay_rate_prev7d      2015
dtype: int64


In [ ]:
print("Dataset shape:", df.shape)

print("\nSplit counts:")
print(df["split"].value_counts(dropna=False))

print("\nTarget columns:")
print([c for c in df.columns if "delay" in c.lower()])

print("\nAll columns:")
print(df.columns.tolist())

Dataset shape: (327346, 25)

Split counts:
split
train    217727
val       55628
test      53991
Name: count, dtype: int64

Target columns:
['delayed', 'origin_delay_rate_prev7d', 'carrier_delay_rate_prev7d', 'dest_delay_rate_prev7d', 'route_delay_rate_prev7d']

All columns:
['date', 'split', 'delayed', 'carrier', 'origin', 'dest', 'route', 'distance', 'dep_time_sin', 'dep_time_cos', 'weekday_sin', 'weekday_cos', 'month_sin', 'month_cos', 'arr_time_sin', 'arr_time_cos', 'origin_flights_prev7d', 'origin_delay_rate_prev7d', 'carrier_flights_prev7d', 'carrier_delay_rate_prev7d', 'dest_flights_prev7d', 'dest_delay_rate_prev7d', 'route_flights_prev7d', 'route_delay_rate_prev7d', 'row_id']


In [ ]:
print("Target distribution by split:")
print(pd.crosstab(df["split"], df["delayed"], normalize="index").round(3))

print("\nAll columns:")
for col in df.columns:
    print(col)

Target distribution by split:
delayed    0.0    1.0
split                
test     0.750  0.250
train    0.744  0.256
val      0.849  0.151

All columns:
date
split
delayed
carrier
origin
dest
route
distance
dep_time_sin
dep_time_cos
weekday_sin
weekday_cos
month_sin
month_cos
arr_time_sin
arr_time_cos
origin_flights_prev7d
origin_delay_rate_prev7d
carrier_flights_prev7d
carrier_delay_rate_prev7d
dest_flights_prev7d
dest_delay_rate_prev7d
route_flights_prev7d
route_delay_rate_prev7d
row_id


In [ ]:
# The dataset has already been loaded in Cell 2. Do not reload it.
print('Input rows:', len(df))

Input rows: 327346


In [ ]:
print(df["split"].value_counts(dropna=False))
print(df[["date", "split"]].head())
print(df["date"].min(), df["date"].max())

split
train    217727
val       55628
test      53991
Name: count, dtype: int64
        date  split
0 2013-01-01  train
1 2013-01-01  train
2 2013-01-01  train
3 2013-01-01  train
4 2013-01-01  train
2013-01-01 00:00:00 2013-12-31 00:00:00


In [ ]:
# Preserve Task 3's month-based chronological split; NEVER create another 70/15/15 split.
df = df.sort_values(['date','row_id']).reset_index(drop=True)
expected = {'train':(1,8), 'val':(9,10), 'test':(11,12)}
for label,(first,last) in expected.items():
    months = df.loc[df['split'].eq(label), 'date'].dt.month
    assert len(months) > 0 and months.between(first,last).all(), f'Incorrect {label} dates'
assert (df.groupby('date')['split'].nunique() == 1).all()
assert df['split'].value_counts().to_dict() == {'train':217727,'val':55628,'test':53991}
display(df.groupby('split')['date'].agg(['min','max']))
print('Original chronological split preserved.')

,min,max
split,,
test,2013-11-01,2013-12-31
train,2013-01-01,2013-08-31
val,2013-09-01,2013-10-31


Original chronological split preserved.


In [ ]:
print('Missing features by column:')
print(df.isna().sum()[lambda s: s > 0])
print('Repeated export-row IDs:', df['row_id'].duplicated().sum())
print('Target values:', sorted(df['delayed'].unique()))
assert df['row_id'].is_unique
assert df['date'].notna().all()
assert df['delayed'].notna().all()
assert set(df['delayed'].unique()).issubset({0,1})
print('Basic checks passed.')

Missing features by column:
origin_delay_rate_prev7d     1759
carrier_delay_rate_prev7d    1768
dest_delay_rate_prev7d       1840
route_delay_rate_prev7d      2015
dtype: int64
Repeated export-row IDs: 0
Target values: [np.float64(0.0), np.float64(1.0)]
Basic checks passed.


In [ ]:
print("Rows:", len(df))
print("Unique dates:", df["date"].nunique())
print("Date range:", df["date"].min(), "to", df["date"].max())

print("\nRows per month:")
print(df.groupby(df["date"].dt.month).size())

print("\nSplit distribution:")
print(df["split"].value_counts())

Rows: 327346
Unique dates: 365
Date range: 2013-01-01 00:00:00 to 2013-12-31 00:00:00

Rows per month:
date
1     26398
2     23611
3     27902
4     27564
5     28128
6     27075
7     28293
8     28756
9     27010
10    28618
11    26971
12    27020
dtype: int64

Split distribution:
split
train    217727
val       55628
test      53991
Name: count, dtype: int64


In [ ]:
# Previous cell already prints this summary; redundant cell removed.

In [ ]:
# Verify chronological separation in the current full dataset
split_dates = df.groupby("split")["date"].agg(["min", "max"])
display(split_dates)

assert split_dates.loc["train", "max"] < split_dates.loc["val", "min"]
assert split_dates.loc["val", "max"] < split_dates.loc["test", "min"]

print("Full-year chronological splits verified!")

,min,max
split,,
test,2013-11-01,2013-12-31
train,2013-01-01,2013-08-31
val,2013-09-01,2013-10-31


Full-year chronological splits verified!


### 3. Chronological Train, Validation, and Test Sets

The dataset already contains chronological split labels created in the previous stage. We validate these labels rather than creating a new random split.

The training set covers January to August 2013, the validation set covers September to October 2013, and the test set covers November to December 2013.

This approach evaluates whether a model trained on earlier flight records can generalise to later periods. It also reduces the risk of overly optimistic results caused by randomly distributing flights from the same time period across different sets.

In [ ]:
df = df.sort_values(['date','row_id']).reset_index(drop=True)
train_df = df[df['split'] == 'train'].copy()
val_df = df[df['split'] == 'val'].copy()
test_df = df[df['split'] == 'test'].copy()
split_summary = (df.groupby('split')
    .agg(flights=('row_id','size'), start_date=('date','min'),
         end_date=('date','max'), delay_rate=('delayed','mean'))
    .reindex(['train','val','test']))
display(split_summary)
assert train_df['date'].max() < val_df['date'].min()
assert val_df['date'].max() < test_df['date'].min()
assert df.groupby('date')['split'].nunique().max() == 1
print('Chronological split validation passed.')

,flights,start_date,end_date,delay_rate
split,,,,
train,217727,2013-01-01,2013-08-31,0.256123
val,55628,2013-09-01,2013-10-31,0.150518
test,53991,2013-11-01,2013-12-31,0.249894


Chronological split validation passed.


### 4. Target Distribution Across Data Splits

The proportion of delayed flights may vary between different periods because of seasonal effects, weather conditions, and changing operational circumstances.

We therefore calculate the target distribution separately for the training, validation, and test sets. This provides context for interpreting subsequent model metrics and helps identify temporal distribution shifts.

The target distribution is reported for analysis only; it is not used to modify the validation or test labels.

In [ ]:
target_summary = (
    df.groupby("split")["delayed"]
    .agg(
        total_flights="count",
        delayed_flights="sum",
        delay_rate="mean"
    )
    .reindex(["train", "val", "test"])
)

target_summary["on_time_rate"] = (
    1 - target_summary["delay_rate"]
)

display(target_summary.round(4))

,total_flights,delayed_flights,delay_rate,on_time_rate
split,,,,
train,217727,55765.0,0.2561,0.7439
val,55628,8373.0,0.1505,0.8495
test,53991,13492.0,0.2499,0.7501


### 5. Feature Selection and Separation of Metadata

We separate the target variable from the predictor variables and define numerical and categorical features explicitly.

The selected predictors follow the feature set recommended in Task 3. Numerical features include distance, cyclic time variables, and historical delay statistics. Categorical features include carrier, origin airport, destination airport, and route.

Identifiers, dates, split labels, and the target are retained for tracking and evaluation but are excluded from the predictor matrix. This ensures that models receive only the intended input features.

In [ ]:
categorical_features = [
    "carrier",
    "origin",
    "dest",
    "route"
]

numeric_features = [
    "distance",
    "dep_time_sin",
    "dep_time_cos",
    "weekday_sin",
    "weekday_cos",
    "month_sin",
    "month_cos",
    "arr_time_sin",
    "arr_time_cos",
    "origin_delay_rate_prev7d",
    "origin_flights_prev7d",
    "carrier_delay_rate_prev7d",
    "carrier_flights_prev7d",
    "dest_delay_rate_prev7d",
    "dest_flights_prev7d",
    "route_delay_rate_prev7d",
    "route_flights_prev7d",
]

model_features = categorical_features + numeric_features
target = "delayed"

assert all(col in df.columns for col in model_features)

assert target not in model_features
assert "id" not in model_features
assert "date" not in model_features
assert "split" not in model_features

print("Categorical features:", len(categorical_features))
print("Numerical features:", len(numeric_features))
print("Total model features:", len(model_features))

display(pd.DataFrame({
    "feature": model_features,
    "type": [
        "Categorical" if c in categorical_features
        else "Numerical"
        for c in model_features
    ]
}))

Categorical features: 4
Numerical features: 17
Total model features: 21


,feature,type
0,carrier,Categorical
1,origin,Categorical
2,dest,Categorical
3,route,Categorical
4,distance,Numerical
5,dep_time_sin,Numerical
6,dep_time_cos,Numerical
7,weekday_sin,Numerical
8,weekday_cos,Numerical
9,month_sin,Numerical


### 6. Creating Predictor Matrices and Target Vectors

We construct separate predictor matrices (`X`) and target vectors (`y`) for training, validation, and testing.

The predictor matrices contain only the selected features, while the target vectors contain the binary delayed-flight label. This separation allows preprocessing to be fitted on the training data without accidentally including the target or evaluation metadata.

The validation set will be used for model selection and threshold decisions, while the test set should be reserved for final evaluation.

In [ ]:
X_train = train_df[model_features].copy()
X_val = val_df[model_features].copy()
X_test = test_df[model_features].copy()

y_train = train_df[target].astype("int64").copy()
y_val = val_df[target].astype("int64").copy()
y_test = test_df[target].astype("int64").copy()

print("X_train:", X_train.shape)
print("X_val:  ", X_val.shape)
print("X_test: ", X_test.shape)

print("\ny_train:", y_train.shape)
print("y_val:  ", y_val.shape)
print("y_test: ", y_test.shape)

X_train: (217727, 21)
X_val:   (55628, 21)
X_test:  (53991, 21)

y_train: (217727,)
y_val:   (55628,)
y_test:  (53991,)


In [ ]:
print("Missing values in training features:")
print(X_train.isna().sum()[X_train.isna().sum() > 0])

print("\nMissing values in validation features:")
print(X_val.isna().sum()[X_val.isna().sum() > 0])

print("\nMissing values in test features:")
print(X_test.isna().sum()[X_test.isna().sum() > 0])

Missing values in training features:
origin_delay_rate_prev7d     1759
carrier_delay_rate_prev7d    1767
dest_delay_rate_prev7d       1812
route_delay_rate_prev7d      1918
dtype: int64

Missing values in validation features:
dest_delay_rate_prev7d      8
route_delay_rate_prev7d    35
dtype: int64

Missing values in test features:
carrier_delay_rate_prev7d     1
dest_delay_rate_prev7d       20
route_delay_rate_prev7d      62
dtype: int64


### 7. Numerical Scaling and Categorical Encoding

Machine learning algorithms can require different representations of input features. Numerical variables are standardised using `StandardScaler`, which learns the mean and standard deviation of each numerical feature from the training set.

Categorical variables are converted into indicator columns using `OneHotEncoder`. The `handle_unknown="ignore"` setting allows the pipeline to process a category that was not observed during training without failing.

Both transformations are combined using a `ColumnTransformer`. The preprocessing pipeline is fitted only on the training data, ensuring that validation and test information does not influence the learned scaling parameters or categorical vocabulary.

In [ ]:
# All learned preprocessing parameters (including imputation) use train only.
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ], remainder='drop', sparse_threshold=1.0,
)
print('Train-only imputation, scaling, and encoding configured.')

Train-only imputation, scaling, and encoding configured.


### 8. Leakage-Safe Preprocessing

We fit the preprocessing pipeline using the training predictors only. The same fitted pipeline is then applied to validation and test predictors.

This distinction is essential: fitting a scaler or encoder on the complete dataset would allow information from future periods to influence the preprocessing stage.

After transformation, all three sets have a consistent feature representation, allowing the modelling team to train on the training set and evaluate on later periods.

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)
from scipy import sparse
for name,m in [('train',X_train_processed),('val',X_val_processed),('test',X_test_processed)]:
    vals = m.data if sparse.issparse(m) else np.asarray(m)
    assert np.isfinite(vals).all(), f'{name} transformed matrix contains NaN/Inf'
    print(name, m.shape)
assert X_train_processed.shape[1] == X_val_processed.shape[1] == X_test_processed.shape[1]

train (217727, 356)
val (55628, 356)
test (53991, 356)


### 9. Inspecting the Processed Feature Representation

After preprocessing, we inspect the number of transformed features and verify that the training, validation, and test matrices have compatible dimensions.

One-hot encoding increases the number of columns because each observed categorical value is represented separately. We also inspect the generated feature names to make the transformed representation easier to understand and document.

In [ ]:
feature_names = preprocessor.get_feature_names_out()

print("Number of processed features:", len(feature_names))
print("\nFirst 30 processed feature names:")
print(feature_names[:30])

print("\nTraining matrix type:", type(X_train_processed))

Number of processed features: 356

First 30 processed feature names:
['num__distance' 'num__dep_time_sin' 'num__dep_time_cos'
 'num__weekday_sin' 'num__weekday_cos' 'num__month_sin' 'num__month_cos'
 'num__arr_time_sin' 'num__arr_time_cos' 'num__origin_delay_rate_prev7d'
 'num__origin_flights_prev7d' 'num__carrier_delay_rate_prev7d'
 'num__carrier_flights_prev7d' 'num__dest_delay_rate_prev7d'
 'num__dest_flights_prev7d' 'num__route_delay_rate_prev7d'
 'num__route_flights_prev7d' 'cat__carrier_9E' 'cat__carrier_AA'
 'cat__carrier_AS' 'cat__carrier_B6' 'cat__carrier_DL' 'cat__carrier_EV'
 'cat__carrier_F9' 'cat__carrier_FL' 'cat__carrier_HA' 'cat__carrier_MQ'
 'cat__carrier_OO' 'cat__carrier_UA' 'cat__carrier_US']

Training matrix type: <class 'scipy.sparse._csr.csr_matrix'>


### 10. Saving the Preprocessing Pipeline and Model Inputs

To make the workflow reproducible, we save the fitted preprocessing pipeline and the transformed datasets. The pipeline contains the scaling parameters and categorical encoding learned from the training set.

The transformed matrices and target vectors can be reused by the modelling team without repeating preprocessing. Saving the pipeline also allows the same transformations to be applied consistently to new flights later.

The test set remains separate and should not be used to fit preprocessing or tune model parameters.

In [ ]:
from scipy import sparse
from pathlib import Path
OUT = Path('/content/task4_outputs')
OUT.mkdir(parents=True, exist_ok=True)
joblib.dump(preprocessor, OUT / 'preprocessor.joblib')
for name,matrix in [('X_train',X_train_processed),('X_val',X_val_processed),('X_test',X_test_processed)]:
    if sparse.issparse(matrix):
        sparse.save_npz(OUT / f'{name}_processed.npz', matrix.tocsr())
    else:
        np.save(OUT / f'{name}_processed.npy', matrix)
for name,y in [('train',y_train),('val',y_val),('test',y_test)]:
    np.save(OUT / f'y_{name}.npy', y.to_numpy())
pd.Series(feature_names, name='feature').to_csv(OUT / 'processed_feature_names.csv',index=False)
for name,part in [('train',train_df),('val',val_df),('test',test_df)]:
    part[['row_id','date','split','delayed']].to_csv(OUT / f'{name}_metadata.csv',index=False)
print('Saved outputs to', OUT)

Saved outputs to /content/task4_outputs


### 11. Deep Learning-Specific Preprocessing

The deep learning model uses a different representation for categorical variables. Instead of one-hot encoding, each categorical value is mapped to an integer index, which can then be passed to a learned embedding layer.

The category mappings are constructed from the training data only. Index 0 is reserved for categories that were not observed during training, ensuring that new categorical values in validation or test data can be handled consistently.

Numerical variables are standardised using a separate scaler fitted only on the training data. This produces numerical and categorical inputs suitable for a compact neural network.

In [ ]:
dl_numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

X_train_num_dl = dl_numeric_pipeline.fit_transform(
    X_train[numeric_features]
).astype(np.float32)

X_val_num_dl = dl_numeric_pipeline.transform(
    X_val[numeric_features]
).astype(np.float32)

X_test_num_dl = dl_numeric_pipeline.transform(
    X_test[numeric_features]
).astype(np.float32)

category_maps = {}

def encode_category(series, mapping):
    return (
        series.astype(str)
        .map(mapping)
        .fillna(0)
        .astype(np.int64)
        .to_numpy()
    )

for col in categorical_features:
    categories = sorted(
        X_train[col].astype(str).unique()
    )

    category_maps[col] = {
        category: index + 1
        for index, category in enumerate(categories)
    }

X_train_cat_dl = np.column_stack([
    encode_category(X_train[col], category_maps[col])
    for col in categorical_features
])

X_val_cat_dl = np.column_stack([
    encode_category(X_val[col], category_maps[col])
    for col in categorical_features
])

X_test_cat_dl = np.column_stack([
    encode_category(X_test[col], category_maps[col])
    for col in categorical_features
])

joblib.dump(
    {
        "numeric_pipeline": dl_numeric_pipeline,
        "category_maps": category_maps,
        "numeric_features": numeric_features,
        "categorical_features": categorical_features,
    },
    OUT / "dl_preprocessing.joblib"
)

for name, matrix in [
    ("X_train_num_dl", X_train_num_dl),
    ("X_val_num_dl", X_val_num_dl),
    ("X_test_num_dl", X_test_num_dl),
    ("X_train_cat_dl", X_train_cat_dl),
    ("X_val_cat_dl", X_val_cat_dl),
    ("X_test_cat_dl", X_test_cat_dl),
]:
    np.save(OUT / f"{name}.npy", matrix)

print("DL numerical inputs:", X_train_num_dl.shape)
print("DL categorical inputs:", X_train_cat_dl.shape)

print("\nCategory vocabulary sizes:")
for col in categorical_features:
    print(col, len(category_maps[col]) + 1,
          "(including unknown index 0)")
for name, m in [('train', X_train_num_dl), ('val', X_val_num_dl), ('test', X_test_num_dl)]:
    assert np.isfinite(m).all(), f'{name} DL numerical inputs contain NaN/Inf'

DL numerical inputs: (217727, 17)
DL categorical inputs: (217727, 4)

Category vocabulary sizes:
carrier 17 (including unknown index 0)
origin 4 (including unknown index 0)
dest 103 (including unknown index 0)
route 219 (including unknown index 0)


### 12. Final Validation and Handoff

In the final stage, we verify that the transformed feature matrices have matching dimensions across the three chronological splits and that the target vectors contain the expected number of records.

We also confirm that the preprocessing pipeline and the deep learning-specific preprocessing artifacts have been saved. The resulting outputs provide the modelling team with consistent inputs while preserving the separation between training, validation, and testing.

The test set must remain reserved for final evaluation after model selection and hyperparameter tuning have been completed.

In [ ]:
assert X_train_processed.shape[0] == len(y_train)
assert X_val_processed.shape[0] == len(y_val)
assert X_test_processed.shape[0] == len(y_test)

assert train_df["date"].max() < val_df["date"].min()
assert val_df["date"].max() < test_df["date"].min()

assert X_train_num_dl.shape[0] == len(y_train)
assert X_val_num_dl.shape[0] == len(y_val)
assert X_test_num_dl.shape[0] == len(y_test)

assert X_train_cat_dl.shape[0] == len(y_train)
assert X_val_cat_dl.shape[0] == len(y_val)
assert X_test_cat_dl.shape[0] == len(y_test)

print("All final checks passed.")
print("Train:", len(y_train))
print("Validation:", len(y_val))
print("Test:", len(y_test))
print("Processed feature count:", len(feature_names))
print("Output directory:", OUT.resolve())

All final checks passed.
Train: 217727
Validation: 55628
Test: 53991
Processed feature count: 356
Output directory: /content/task4_outputs


In [ ]:
# Export processed matrices in chunks, avoiding full sparse -> dense conversion.
from scipy import sparse
from google.colab import files

filename = '/content/Task4_Preprocessing.csv'

def append_split_csv(matrix, metadata, write_header, chunk_size=1500):
    for start in range(0, matrix.shape[0], chunk_size):
        end = min(start+chunk_size, matrix.shape[0])
        chunk = matrix[start:end]
        if sparse.issparse(chunk):
            chunk = chunk.toarray()
        feats = pd.DataFrame(chunk, columns=feature_names)
        info = metadata.iloc[start:end][['row_id','date','split','delayed']].reset_index(drop=True)
        block = pd.concat([info, feats], axis=1)
        block.to_csv(filename, mode='w' if write_header else 'a',
                     header=write_header, index=False)
        write_header = False
    return write_header

header = True
for matrix,metadata in [(X_train_processed,train_df),
                        (X_val_processed,val_df),
                        (X_test_processed,test_df)]:
    header = append_split_csv(matrix,metadata,header)
print('Saved processed CSV:',filename)
files.download(filename)

Saved processed CSV: /content/Task4_Preprocessing.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from scipy import sparse
for name,matrix in [('train',X_train_processed),('val',X_val_processed),('test',X_test_processed)]:
    values = matrix.data if sparse.issparse(matrix) else np.asarray(matrix)
    assert np.isfinite(values).all(), f'{name} has non-finite processed values'
print('No missing or infinite values in processed model inputs.')

No missing or infinite values in processed model inputs.
